In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("practice10.ipynb")

**Student names and e-mails:**

_YOUR NAME — your@calvin.edu_

_YOUR NAME — your@calvin.edu_

# Practice 10 — Rare Legends and Daily Births: Imbalanced and Time-Series Data

Two datasets this time, one for each of this week's classes:

* **Part 1:** 721 Pokémon, of which only 46 are *legendary*. Can a model find the rare ones, and what does "accurate" mean when they are rare?
* **Part 2:** every day's number of births in the United States, 2000 to 2014. How do we forecast a day we haven't seen yet, and test the forecast honestly?

| SLO | Description |
|-----|-------------|
| **12A** | Identify class imbalance in a dataset and apply techniques such as resampling or adjusted decision thresholds to address it |
| **12B** | Engineer features for time-series data (lag features, rolling statistics) and use time-aware train-test splits to avoid data leakage |

### Setup (given — just run it)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_predict, cross_validate, cross_val_score, KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, recall_score, precision_score, mean_absolute_error

# imbalanced-learn (imblearn) comes with Colab; elsewhere: pip install imbalanced-learn
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.ensemble import BalancedRandomForestClassifier

---
## Part 1 — Legendary Pokémon (SLO 12A)

In the Pokémon games, a few creatures are **legendary**: rare, powerful, usually one of a kind. This dataset has the first six generations of Pokémon (the games from 1996 to 2013), one row per species, with the six **base stats** the games use in battle.

| Column | What it holds |
|---|---|
| `name` | the Pokémon |
| `type1`, `type2` | its type(s), like Fire or Water (`type2` is blank for single-type Pokémon) |
| `generation` | the game generation that introduced it, 1 to 6 |
| `hp` | hit points: how much damage it can take |
| `attack`, `defense` | strength of its physical moves, and resistance to them |
| `sp_attack`, `sp_defense` | the same for special moves (like fire or psychic attacks) |
| `speed` | who moves first in a battle |
| `legendary` | **the label**: 1 = legendary, 0 = not |

<sub>Data: A. Barradas, "Pokemon with Stats" (Kaggle, 2016), CC0, one row per species (Mega Evolutions and other alternate forms removed). Pokémon and Pokémon names are trademarks of Nintendo, Creatures and Game Freak.</sub>

In [ ]:
pokemon = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/pokemon.csv")
print(pokemon["legendary"].value_counts())
print(f"share of legendary Pokémon: {pokemon['legendary'].mean():.1%}")
pokemon.head()

46 of 721: **6.4%**. This is **imbalanced data**: the majority class (ordinary Pokémon) outnumbers the minority (legendary) about 15 to 1.

The features are the six stats. The split is given. `stratify=y` keeps the share of legendaries the same in the training and the test set: with only 46 of them, a plain random split could put 8 or 16 in the test set just by luck.

In [ ]:
stats = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]
X = pokemon[stats]
y = pokemon["legendary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
print(f"training: {len(y_train)} Pokémon, {y_train.sum()} legendary | test: {len(y_test)} Pokémon, {y_test.sum()} legendary")

lazy = np.zeros(len(y_test))          # a lazy "model": nobody is legendary
print(f"lazy model | accuracy {accuracy_score(y_test, lazy):.3f} | recall {recall_score(y_test, lazy):.3f}")

The lazy model is **93.4%** accurate and finds **none** of the 12 test legendaries. Accuracy rewards guessing the majority, so from now on we watch **recall** (of the real legendaries, the share the model finds) and **precision** (of the Pokémon it calls legendary, the share that really are).

### Task 12A.1 — Accurate, but Is It Finding Them? *(2 pts)*

1. Build a pipeline that scales the stats, then runs logistic regression: `make_pipeline(StandardScaler(), LogisticRegression())`. Store it in `model` and fit it on the training Pokémon.
2. Store its predictions for the test Pokémon in `y_pred`.
3. Store its test **accuracy**, **recall** and **precision** in `acc`, `rec` and `prec`. (*Hint:* `accuracy_score(y_test, y_pred)`, and the same for `recall_score` and `precision_score`.)

The printout is given.

In [ ]:
model = ...
...
y_pred = ...
acc = ...
rec = ...
prec = ...

print(f"logistic regression | accuracy {acc:.3f} | recall {rec:.3f} | precision {prec:.3f} | called legendary: {y_pred.sum()}")
test_pokemon = pokemon.loc[X_test.index].assign(total=X_test.sum(axis=1), called=y_pred)
test_pokemon[test_pokemon["legendary"] + test_pokemon["called"] > 0][["name", "total", "legendary", "called"]].sort_values("total")

In [ ]:
grader.check("12A.1")

**93.9%** accurate, half a point above the lazy model, and it finds only **4 of the 12** legendaries (recall 0.33).

The table (given) shows each test Pokémon that is legendary or was called legendary, with its **total**, the sum of its six stats. The four it catches are giants: Giratina, Reshiram, Xerneas and Yveltal, each with a total of 680. The eight it misses (Articuno, Raikou, Regirock, Regice, Uxie, Mesprit, Azelf, Terrakion) all have 580, close to the strongest ordinary Pokémon: Hydreigon (600), Manaphy (600) and Slaking (670) are its three false alarms.

That's the lean toward the majority. The eight missed legendaries get probabilities between **0.23 and 0.48**: far above the 6% of legendaries overall, but not "more likely than not". Logistic regression learned from 506 ordinary Pokémon and only 34 legendary ones, and it keeps its mistakes, counted over all of them, as few as it can. With 15 ordinary Pokémon for every legendary, every probability gets pulled toward "not legendary". Uxie (0.476) and Raikou (0.474) miss the 0.5 cut by less than 0.03.

---
### Moving the Threshold

`predict` says "legendary" when the probability is at least **0.5**. That threshold is a knob, so we choose it with cross-validation on the **training** Pokémon, never by peeking at the test set. `cross_val_predict` gives each training Pokémon a probability from a model that didn't learn from it.

### Task 12A.2 — Choose a Threshold *(2 pts)*

1. Store each training Pokémon's cross-validated probability of being legendary in `p_train`: `cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1]`. (Column `1` is the probability of class 1, legendary.) The table of thresholds is given.
2. The table shows that **0.2** catches almost every legendary at about the same precision. Use it on the test set: the model is already fit, so store `model.predict_proba(X_test)[:, 1] >= 0.2` in `called`, then the test recall and precision of `called` in `rec_02` and `prec_02`.

In [ ]:
p_train = ...

rows = []
for t in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    flagged = p_train >= t
    rows.append({"threshold": t, "called legendary": flagged.sum(),
                 "recall": recall_score(y_train, flagged), "precision": precision_score(y_train, flagged)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

called = ...
rec_02 = ...
prec_02 = ...
print(f"\ntest set, threshold 0.2 | recall {rec_02:.3f} | precision {prec_02:.3f} | called legendary: {called.sum()}")

In [ ]:
grader.check("12A.2")

In cross-validation, lowering the threshold from 0.5 to **0.2** raises recall from 0.65 to **0.97**, and precision barely moves (0.73 to 0.70). The legendaries' stats do stand out; the 0.5 line was just in the wrong place. Below 0.2, precision drops fast: at 0.05, more than half of the Pokémon called legendary are not.

On the test set, 0.2 finds **all 12** legendaries, with 4 false alarms out of 16 (precision 0.75): Hydreigon, Manaphy, Slaking and Volcarona.

> 📌 **One split:** 12 test legendaries. One legendary is more than 8 points of recall.

---
### Rebalancing, for a Model Without Weights

kNN (Week 6) has no `class_weight` option, and with 15 ordinary Pokémon for every legendary, most of a legendary's nearest neighbors are ordinary (given):

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
knn_scores = cross_validate(knn, X_train, y_train, cv=5, scoring=["recall", "precision"])
print(f"kNN, k = 5 | recall {knn_scores['test_recall'].mean():.3f} | precision {knn_scores['test_precision'].mean():.3f}")

Recall **0.52**: kNN misses about half the legendaries. SMOTE can help: it makes new, synthetic legendaries on the line between a real legendary and one of its nearest legendary neighbors, until both classes are the same size.

### Task 12A.3 — SMOTE for kNN *(2 pts)*

1. Build an **imblearn** pipeline that scales, applies SMOTE, then runs kNN: `make_imb_pipeline(StandardScaler(), SMOTE(random_state=1), KNeighborsClassifier())`. Store it in `knn_smote`.
2. Score it with `cross_validate` on the training Pokémon, with `cv=5` and `scoring=["recall", "precision"]`. Store the result in `knn_smote_scores`.

The printout is given.

In [ ]:
knn_smote = ...
knn_smote_scores = ...

print(f"kNN + SMOTE | recall {knn_smote_scores['test_recall'].mean():.3f} | precision {knn_smote_scores['test_precision'].mean():.3f}")

In [ ]:
grader.check("12A.3")

Recall **0.52 → 0.97**, precision 0.90 → 0.63: SMOTE turned kNN from careful into thorough.

Why `make_imb_pipeline` and not scikit-learn's `make_pipeline`? In imblearn's pipeline, a sampler runs only when the pipeline **learns** (`fit`), never when it predicts. So in each round of cross-validation, synthetic legendaries are made from the four folds it learns from, and the held-out fold keeps the real 6% of legendaries. SMOTE-ing all the data first would put synthetic copies of held-out legendaries into the training folds: a leak.

The whole toolbox from class, on the same folds (given — about 15 seconds):

In [ ]:
options = {
    "logistic, nothing": make_pipeline(StandardScaler(), LogisticRegression()),
    "logistic, class weights": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")),
    "logistic, oversample": make_imb_pipeline(StandardScaler(), RandomOverSampler(random_state=1), LogisticRegression()),
    "logistic, undersample": make_imb_pipeline(StandardScaler(), RandomUnderSampler(random_state=1), LogisticRegression()),
    "logistic, SMOTE": make_imb_pipeline(StandardScaler(), SMOTE(random_state=1), LogisticRegression()),
    "kNN, nothing": knn,
    "kNN, SMOTE": knn_smote,
    "forest, nothing": RandomForestClassifier(random_state=1),
    "forest, class weights": RandomForestClassifier(class_weight="balanced", random_state=1),
    # imblearn's newer defaults, written out so every version builds the same forest
    "balanced forest": BalancedRandomForestClassifier(sampling_strategy="all", replacement=True, bootstrap=False, random_state=1),
}
rows = []
for name, option in options.items():
    scores = cross_validate(option, X_train, y_train, cv=5, scoring=["recall", "precision"])
    rows.append({"option": name, "recall": scores["test_recall"].mean(), "precision": scores["test_precision"].mean()})
pd.DataFrame(rows).round(3)

Read it against the class toolbox:

* **Logistic regression:** class weights, oversampling and SMOTE all land near recall 0.94–0.97 and precision 0.66–0.69, about where the threshold 0.2 put us. For logistic regression, rebalancing mostly slides the line, like moving the threshold.
* **Undersampling** reaches the same recall, but precision falls to **0.47**. With 34 training legendaries, it keeps only 34 ordinary Pokémon too: 68 Pokémon to learn from. Undersampling is for a huge majority, not for small data.
* **Forest, class weights:** 0.59 recall, the same as with nothing. A fully grown tree ends in pure leaves, so weighting the rows barely changes its votes. The **balanced forest**, where each tree learns from its own balanced sample, does move: recall 0.82.

A fix that helps one model can do nothing for another: compare them with cross-validation.

### Who Decides What "Legendary" Means?

The label comes from the original file, and it doesn't follow a single rule. The games have a separate category, **mythical** Pokémon, usually given out at special events. In this file, Mew, Celebi and Manaphy (mythical) are labeled **not** legendary, while Jirachi, Arceus and Darkrai (also mythical) are labeled **legendary**. So one of the model's "false alarms", Manaphy, is a Pokémon that some fans, and some files, would call legendary. Before trusting a recall or precision, ask what the rare class really is, and who decided.

---
## Part 2 — Daily Births (SLO 12B)

How many babies will be born in the United States next Tuesday? Hospitals plan staff and beds around questions like that. The U.S. Social Security Administration counts the births of every day, and FiveThirtyEight shared the counts for 2000 to 2014, for a story on whether fewer babies are born on Friday the 13th.

* `date`: the day
* `births`: babies born in the United States that day

<sub>Data: FiveThirtyEight, "U.S. Births" (US_births_2000-2014_SSA.csv), counts from the Social Security Administration, CC BY 4.0.</sub>

In [ ]:
births = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/births.csv", parse_dates=["date"])
births["day_of_week"] = births["date"].dt.dayofweek       # 0 = Monday ... 6 = Sunday
births["day_of_year"] = births["date"].dt.dayofyear
births["month"] = births["date"].dt.month
births["year"] = births["date"].dt.year

px.line(births, x="date", y="births", title="Births per Day in the United States, 2000–2014",
        labels={"date": "", "births": "births per day"}).show()
births.groupby("day_of_week")["births"].mean().round().rename(index=dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])))

The line looks like two bands, because the days of the week differ so much: from about **11,900** (Monday) to **13,100** (Tuesday) births on an average weekday, but **8,563** on an average Saturday and **7,518** on a Sunday. Many births are scheduled (inductions and planned C-sections), and doctors schedule them on weekdays. The lowest days of all are Christmas Days. And, yes: an average Friday the 13th had 11,950 births, 6% fewer than the Fridays a week before and after it.

Across the years, births per day rose to a peak in **2007** (12,002 on average) and then fell (10,931 in 2012).

### Task 12B.1 — Lag Features and a Naive Forecast *(2 pts)*

A **lag feature** moves a past value of the label down into the current row, with `shift`.

1. Add four columns to `births`:
   * `births_1d_ago`: the births of the day before, `births["births"].shift(1)`
   * `births_7d_ago`: the same weekday one week before, `shift(7)`
   * `births_364d_ago`: the same weekday one year (52 weeks) before, `shift(364)`
   * `mean_last_7`: the average of the 7 days before this one, `births["births"].shift(1).rolling(7).mean()`
2. (given) Drop the rows whose lags reach before 2000, and split by date: **learn from 2000–2009, forecast 2010–2014**.
3. A **naive forecast** copies the past, with no model. Store the test MAE of "the same as one week ago" in `naive_mae`: `mean_absolute_error(y[test], births.loc[test, "births_7d_ago"])`.

The comparison of all the naive forecasts is given.

In [ ]:
births["births_1d_ago"] = ...
births["births_7d_ago"] = ...
births["births_364d_ago"] = ...
births["mean_last_7"] = ...

births = births.dropna().reset_index(drop=True)     # the first 364 days have no lag from a year before
y = births["births"]
learn = births["date"] < "2010-01-01"               # learn from the past...
test = ~learn                                       # ...forecast 2010-2014
print(f"learn: {learn.sum()} days | test: {test.sum()} days")

naive_mae = ...

for lag in ["births_1d_ago", "births_7d_ago", "births_364d_ago", "mean_last_7"]:
    print(f"naive forecast, copy {lag:16} | test MAE: {mean_absolute_error(y[test], births.loc[test, lag]):.0f} births")

In [ ]:
grader.check("12B.1")

* **The day before: 1,698.** A Monday copies a Sunday, a Saturday copies a Friday. The nearest past isn't always the most useful.
* **A week before: 452.** The same weekday: the weekly cycle lines up.
* **The same weekday a year before: 378,** the best of the four. It lines up the week **and** the season.
* **The average of the last 7 days: 2,004.** It smooths away the weekday pattern, the most important thing here.

Pick lags at the data's cycles. And a model that can't beat 378 isn't worth running.

### Task 12B.2 — Calendar Only, or Calendar and Lags? *(2 pts)*

1. Fit a `RandomForestRegressor(random_state=1)` on the training days with the `calendar` columns only (given list), and store its test MAE in `mae_calendar`.
2. Fit the same forest with `calendar + lags`, and store its test MAE in `mae_lags`.

(*Hint:* `forest.fit(births.loc[learn, columns], y[learn])`, then `mean_absolute_error(y[test], forest.predict(births.loc[test, columns]))`.) The plot is given.

In [ ]:
calendar = ["day_of_week", "day_of_year", "month"]
lags = ["births_1d_ago", "births_7d_ago", "births_364d_ago", "mean_last_7"]
forest = RandomForestRegressor(random_state=1)

...
calendar_pred = forest.predict(births.loc[test, calendar])
mae_calendar = ...

...
lags_pred = forest.predict(births.loc[test, calendar + lags])
mae_lags = ...

print(f"forest, calendar only   | test MAE: {mae_calendar:.0f} | too high by {(calendar_pred - y[test]).mean():.0f} births a day, on average")
print(f"forest, calendar + lags | test MAE: {mae_lags:.0f} | too high by {(lags_pred - y[test]).mean():.0f} births a day, on average")
weekly = births.loc[test, ["date", "births"]].assign(**{"calendar only": calendar_pred, "calendar + lags": lags_pred})
px.line(weekly.resample("W", on="date").mean(), y=["births", "calendar only", "calendar + lags"],
        title="2010–2014, Averaged by Week: What Happened and the Two Forecasts",
        labels={"value": "births per day", "date": "", "variable": ""}).show()

In [ ]:
grader.check("12B.2")

**Calendar only: 626**, and too high by **595** births a day on average. For each date, the forest predicts about the average of 2000–2009 for that kind of day, 11,556 births a day, but 2010–2014 averaged 10,979. Births had been falling since 2007, and a forest can't forecast a level it has never seen.

**Calendar and lags: 299**, better than the best naive forecast (378). The lags carry the current **level** into each row: if births have been lower lately, the forecast is lower. It still runs a little high (+112), because the decline kept going.

On the days it learned from, the lag forest misses by only 103: the usual gap between memorized rows and new ones. Only the test number tells how it forecasts.

> 📌 **One split:** one stretch of five years. Another stretch would give other numbers.

### The Week on a Circle? (given)

In class, the hour went on a circle so that 11 pm sat next to midnight. Should the **day of the week** go on a circle, so that Sunday sits next to Monday? Three versions of the same linear regression, with the day of the week as a plain number, on a circle, and as categories (one column per day, Week 9's one-hot):

> 🎯 **Predict First:** Which will forecast best?

In [ ]:
births["dow_sin"] = np.sin(2 * np.pi * births["day_of_week"] / 7)      # a full turn = 7 days
births["dow_cos"] = np.cos(2 * np.pi * births["day_of_week"] / 7)
versions = {"a number": births[["day_of_week"]],
            "a circle": births[["dow_sin", "dow_cos"]],
            "categories": pd.get_dummies(births["day_of_week"], prefix="day", drop_first=True)}
table = {"actual, 2010-2014": y[test].groupby(births.loc[test, "day_of_week"]).mean()}
for name, columns in versions.items():
    line = LinearRegression().fit(columns[learn], y[learn])
    predicted = pd.Series(line.predict(columns[test]), index=columns[test].index)
    print(f"day of week as {name:10} | test MAE: {mean_absolute_error(y[test], predicted):.0f}")
    table[name] = predicted.groupby(births.loc[test, "day_of_week"]).mean()
pd.DataFrame(table).round().rename(index=dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])))

* **A number (1,386):** a straight line, falling from Monday (14,009) to Sunday (9,102). Monday is too high, the weekend not low enough.
* **A circle (1,250):** barely better. On the circle, Sunday is Monday's neighbor, so the wave drags Monday down toward Sunday (10,664 against 11,709 actual) and puts the peak on Wednesday.
* **Categories (689):** each day gets its own level, and the weekend drop is right where it belongs.

A circle says "neighbors on the cycle behave alike". That's true for hours of the day and days of the year; for births, it isn't true of Sunday and Monday. Ask before you use it. (For a forest, plain numbers are fine: a tree can split off the weekend with one question, "day of week ≥ 5?")

---
### Testing in Time Order

To choose features and knobs we need cross-validation on 2000–2009, and it must not touch 2010–2014. `KFold(shuffle=True)` deals the days into five folds at random. `TimeSeriesSplit` **replays the past** instead: learn from everything before a date, check the stretch right after it, move the date forward.

Here is a forest that knows only the **date**: day of week, day of year, month and **year**.

> 🎯 **Predict First:** Which cross-validation will promise the smaller error: shuffled or in time order?

### Task 12B.3 — Shuffled Folds or Rolling Folds? *(2 pts)*

`X_dev` and `y_dev` (given) are the training days, 2000–2009.

1. Store the five fold MAEs of `RandomForestRegressor(random_state=1)` on `X_dev[date_only]` with **shuffled** folds in `shuffled_mae`: `-cross_val_score(..., cv=KFold(n_splits=5, shuffle=True, random_state=1), scoring="neg_mean_absolute_error")`. (scikit-learn maximizes scores, so it reports the MAE as a negative number; the minus sign turns it back.)
2. Do the same with `cv=TimeSeriesSplit(n_splits=5)`, and store the result in `rolling_mae`.

The fold dates, the printout and the real test are given.

In [ ]:
X_dev, y_dev = births.loc[learn], y[learn]
date_only = ["day_of_week", "day_of_year", "month", "year"]
for fold, (learn_rows, check_rows) in enumerate(TimeSeriesSplit(n_splits=5).split(X_dev)):
    print(f"rolling fold {fold + 1}: learn {X_dev['date'].iloc[learn_rows[0]]:%b %Y} to {X_dev['date'].iloc[learn_rows[-1]]:%b %Y}"
          f" | check {X_dev['date'].iloc[check_rows[0]]:%b %Y} to {X_dev['date'].iloc[check_rows[-1]]:%b %Y}")

shuffled_mae = ...
rolling_mae = ...

print("\nshuffled folds:", shuffled_mae.round(), f"| mean {shuffled_mae.mean():.0f}")
print("rolling folds: ", rolling_mae.round(), f"| mean {rolling_mae.mean():.0f}")
date_forest = RandomForestRegressor(random_state=1).fit(X_dev[date_only], y_dev)
print(f"the real test, 2010-2014: {mean_absolute_error(y[test], date_forest.predict(births.loc[test, date_only])):.0f}")

In [ ]:
grader.check("12B.3")

* **Shuffled folds: 277**, steady in every fold (238 to 323). That would make the date forest better than the forest with lags (299)! It isn't. With shuffled folds, every checked day has its neighbors among the days the forest learned from: the day before and after, the same weekday a week earlier and later, all in the same year. With `year` and `day_of_year`, the forest finds them and copies their births. It isn't forecasting; it's **peeking**, like scoring a model on rows it learned from (Week 6).
* **Rolling folds: 410**, from 306 to 586. Each checked stretch comes after everything the forest learned from, as in real use. Fold 5 checked July 2008 to 2009, the start of the decline, and missed by **586**.
* **The real test: 595.** The future did what fold 5 warned about. The year column didn't help: a tree can't forecast a year it has never seen, so it treats 2010–2014 like 2009.

For a time series, use `TimeSeriesSplit` in `cross_val_score` and in `GridSearchCV`, and report the folds' **range**, not one number. For the forest with lags (given):

In [ ]:
features = calendar + lags
rolling = -cross_val_score(RandomForestRegressor(random_state=1), X_dev[features], y_dev, cv=TimeSeriesSplit(n_splits=5), scoring="neg_mean_absolute_error")
print("forest with lags, rolling folds:", rolling.round(), f"| range {rolling.min():.0f} to {rolling.max():.0f} | real test: {mae_lags:.0f}")

Rolling folds: **280 to 386**. The real test, 299, landed inside that range: an honest promise.

### The Days the Lags Get Wrong (given)

Where does the lag forest miss most?

In [ ]:
misses = births.loc[test, ["date", "births"]].assign(forecast=lags_pred.round(), miss=(lags_pred - y[test]).abs().round())
misses["day"] = misses["date"].dt.strftime("%a %b %d, %Y")
misses.nlargest(10, "miss")[["day", "births", "forecast", "miss"]]

Nine of the ten are **holidays**: Memorial Day, Christmas, Thanksgiving, the Fourth of July (and July 5, 2010, when it was observed). Scheduled births move away from them. The tenth is **Monday, July 2, 2012**: the forecast was about 2,700 too low, because "the same weekday a year before", 364 days earlier, was Monday, July 4, 2011, a holiday. Holidays move around the calendar, so a 364-day lag carries last year's holiday onto an ordinary day. A `holiday` column would teach the forest what the calendar alone can't.

Those dips are people's choices, not biology: births are scheduled around weekends, holidays and even Friday the 13th. Like Stone's Fitbit wearers, the parents and doctors behind these numbers are people who think and plan, and the time series records their calendars as much as nature's.

---
## Submission

Save your notebook, then upload the **`.ipynb` file** directly to our Moodle assignment page. Don't export, zip, or submit anything else — just the notebook.